
# Optimized temporal MS-AA participant-held-out top-\(m\) analysis

This notebook implements the **same participant-held-out cross-fit logic** as the
canonical spatial analysis, but is optimized for the temporal MS-AA representation.

For each condition, model order \(K\), and outer split:

1. participants are split into halves A and B;
2. individual archetypes are ranked using A only;
3. top-\(m\) reconstructions are evaluated using B only;
4. the direction is reversed (B ranks, A evaluates);
5. the procedure is repeated across 20 outer participant splits.

## What is optimized

The statistical procedure is unchanged. The speedup comes from algebra and caching:

- **Single-archetype ranking:** a normalized single temporal archetype is rank-1.
  The decoder is therefore evaluated in low-rank form without repeatedly constructing
  full \(T\times V\) reconstructions.
- **Top-\(m\) evaluation:** raw archetype contributions are accumulated once in ranked
  order. At each requested \(m\), the reconstruction is normalized and all 20 inner
  participant partitions are evaluated in one batched matrix operation.
- **Matched-ratio scope:** by default this notebook runs only the 12 temporal \(K\)
  values needed to reproduce the compact spatial-vs-temporal heatmap.

The outputs are compatible with the previous temporal held-out notebook and are
written to:

`msaa_heldout_topm_outputs_temporal_across_fullgrid/`

The notebook is resumable and will reuse any valid rows already present there.


In [ ]:

from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from msaa_pipeline.decode import get_subjects, decode_subjects

ANALYSIS_TYPE = "temporal"
FIT_SCOPE = "across"
OUTPUT_ROOT = Path(".")
FIT_DIR = Path("msaa_flexible_outputs_npz")

OUTPUT_DIR = Path("msaa_heldout_topm_outputs_temporal_across_fullgrid")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_CSV = OUTPUT_DIR / "heldout_topm_crossfit_runs.csv"
RANK_CSV = OUTPUT_DIR / "heldout_per_archetype_ranking_runs.csv"
SUMMARY_CSV = OUTPUT_DIR / "heldout_topm_summary.csv"
SEED_CSV = OUTPUT_DIR / "outer_split_seeds.csv"

CONDITIONS = ["intact", "word", "rest"]
CONDITION_INDEX = {c: i for i, c in enumerate(CONDITIONS)}

TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 40, 50, 75, 100]

# Exact 12 temporal model orders used by the compact legacy heatmap:
# S7/T3, S21/T9, S44/T18, S46/T20, S53/T23, S58/T25,
# S63/T27, S88/T38, S126/T54, S200/T90, S300/T129, S700/T300.
COMPACT_TEMPORAL_K = [30,45,60,75,81,108]

N_OUTER_REPS = 20
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42
EPS = 1e-8

# Change to False only if you later want the full historical custom matched-K grid.
RUN_COMPACT_HEATMAP_K_ONLY = True

# Print progress every this many ranked archetypes.
RANK_PROGRESS_EVERY = 10

print("Output:", OUTPUT_DIR.resolve(), flush=True)


In [ ]:

def discover_temporal_k():
    pat = re.compile(r"^temporalAA_across_acrossCond_K(\d+)\.npz$")
    ks = []
    for p in FIT_DIR.glob("*.npz"):
        m = pat.match(p.name)
        if m:
            ks.append(int(m.group(1)))
    return sorted(set(ks))

available_K = discover_temporal_k()
if not available_K:
    raise FileNotFoundError(
        f"No temporal across-condition fits found under {FIT_DIR.resolve()}"
    )

if RUN_COMPACT_HEATMAP_K_ONLY:
    K_VALUES = [k for k in COMPACT_TEMPORAL_K if k in available_K]
    missing = sorted(set(COMPACT_TEMPORAL_K) - set(K_VALUES))
    if missing:
        print("WARNING: required compact temporal K values missing:", missing, flush=True)
else:
    K_VALUES = available_K

print("Temporal K values to run:", K_VALUES, flush=True)



## Exact decoder helpers

These functions reproduce the historical two-fold, repeated across-participant
timepoint decoder, including the original definitions of accuracy, normalized temporal
error, and rank.

For `nfolds=2`, fold 1 is the transpose of fold 0's correlation matrix. We therefore
compute each random participant partition only once and evaluate both directions from
that matrix.


In [ ]:

def get_xval_assignments(ndata, nfolds, rng):
    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))
    for i in range(1, nfolds):
        inds = np.arange(i * groupsize, min((i + 1) * groupsize, ndata))
        group_assignments[inds] = i
    rng.shuffle(group_assignments)
    return group_assignments


def decoder_metrics(corrs):
    """Vectorized equivalent of the historical decoder(corrs)."""
    T = corrs.shape[0]
    true_t = np.arange(T)
    pred = np.argmax(corrs, axis=1)

    accuracy = float(np.mean(pred == true_t))
    # Historical code divided by T inside the loop and again afterward.
    error = float(np.mean(np.abs(pred - true_t)) / T)

    diag = np.diag(corrs)
    rank = float(np.mean(np.mean(corrs <= diag[:, None], axis=1)))

    return {
        "accuracy": accuracy,
        "error": error,
        "rank": rank,
    }


def summarize_metric_rows(rows):
    df = pd.DataFrame(rows)
    return {
        "mean": float(df["accuracy"].mean()),
        "err": float(df["accuracy"].std(ddof=1) / np.sqrt(max(len(df), 1))),
        "std": float(df["accuracy"].std(ddof=1)),
        "count": int(len(df)),
        "rank_mean": float(df["rank"].mean()),
        "error_mean": float(df["error"].mean()),
    }


def inner_masks(n_subjects, nreps, seed):
    """
    Return the fold-0 mask for every repetition.
    Fold 1 is its complement and uses the transposed correlation matrix.
    """
    if NFOLDS_DECODE != 2:
        raise ValueError("Optimized decoder currently assumes NFOLDS_DECODE=2.")

    rng = np.random.default_rng(seed)
    masks = np.zeros((nreps, n_subjects), dtype=bool)

    for rep in range(nreps):
        fold_ids = get_xval_assignments(n_subjects, 2, rng)
        masks[rep] = (fold_ids == 0)

    return masks


def batch_decode_stack(recon_stack, nreps=20, seed=42):
    """
    Exact batched equivalent of run_timepoint_decoding for nfolds=2.

    recon_stack: (N, T, V), already normalized within participant/node over time.
    """
    X = np.asarray(recon_stack, dtype=float)
    N, T, V = X.shape

    masks0 = inner_masks(N, nreps, seed)
    n0 = masks0.sum(axis=1).astype(float)
    n1 = (N - masks0.sum(axis=1)).astype(float)

    w0 = masks0 / n0[:, None]
    w1 = (~masks0) / n1[:, None]

    # Group means for every repetition at once: (R, T, V)
    A = np.einsum("rn,ntv->rtv", w0, X, optimize=True)
    B = np.einsum("rn,ntv->rtv", w1, X, optimize=True)

    # Pearson correlation across features for every T x T pair.
    A = A - A.mean(axis=2, keepdims=True)
    B = B - B.mean(axis=2, keepdims=True)

    normA = np.linalg.norm(A, axis=2)
    normB = np.linalg.norm(B, axis=2)

    A = A / np.where(normA[:, :, None] > 0, normA[:, :, None], 1.0)
    B = B / np.where(normB[:, :, None] > 0, normB[:, :, None], 1.0)

    corrs = np.matmul(A, np.swapaxes(B, 1, 2))
    corrs = np.nan_to_num(corrs, nan=0.0, posinf=0.0, neginf=0.0)

    rows = []
    for rep in range(nreps):
        m0 = decoder_metrics(corrs[rep])
        m0.update({"rep": rep, "fold": 0})
        rows.append(m0)

        # fold 1 simply reverses the two participant groups
        m1 = decoder_metrics(corrs[rep].T)
        m1.update({"rep": rep, "fold": 1})
        rows.append(m1)

    return summarize_metric_rows(rows)



## Fast exact ranking of single temporal archetypes

For temporal MS-AA and one archetype \(k\),

\[
X^{(s)}_k(t,v)=S^{(s)}_{k,t}\,\mathrm{sXC}^{(s)}_{v,k}.
\]

After the same per-node temporal z-scoring used by the original pipeline, this remains
an outer product. We exploit that rank-1 form to compute the correlation matrices
without materializing a \(T\times V\) reconstruction for every archetype and every
participant split.

The synthetic equivalence test used when developing this notebook matched the
historical decoder to floating-point precision for accuracy, rank, and temporal error.


In [ ]:

def temporal_single_factors(subjects, archetype):
    """
    Exact factors for the normalized reconstruction of one temporal archetype.

    Returns
    -------
    U : (N, T)
    W : (N, V)
    G : (N, N)
        Feature-centered spatial-factor Gram matrix.
    """
    U = []
    W = []

    for sub in subjects:
        sXC = np.asarray(sub["sXC"], dtype=float)  # V x K
        S = np.asarray(sub["S"], dtype=float)      # K x T

        temporal = S[archetype, :]
        spatial = sXC[:, archetype]

        centered_t = temporal - temporal.mean()
        sd_t = temporal.std()

        # Exact consequence of z-scoring the outer product across time per node:
        # (spatial * centered_t) / (abs(spatial)*sd_t + eps)
        spatial_factor = spatial / (np.abs(spatial) * sd_t + EPS)

        U.append(centered_t)
        W.append(spatial_factor)

    U = np.asarray(U, dtype=float)
    W = np.asarray(W, dtype=float)

    # Pearson correlation later centers rows across V. Center each subject's
    # spatial factor here so group means can be evaluated in low-rank form.
    Wc = W - W.mean(axis=1, keepdims=True)
    G = Wc @ Wc.T

    return U, W, G


def lowrank_single_decode_summary(U, G, subset_indices, nreps, seed):
    """
    Decode a single temporal archetype on an arbitrary participant subset
    using only the T x N temporal factors and the N x N spatial Gram matrix.
    """
    subset_indices = np.asarray(subset_indices, dtype=int)
    n = len(subset_indices)

    rng = np.random.default_rng(seed)
    rows = []

    for rep in range(nreps):
        fold_ids = get_xval_assignments(n, 2, rng)

        ia = subset_indices[fold_ids == 0]
        ib = subset_indices[fold_ids == 1]

        Ua = U[ia].T  # T x nA
        Ub = U[ib].T  # T x nB

        Gaa = G[np.ix_(ia, ia)] / (len(ia) ** 2)
        Gbb = G[np.ix_(ib, ib)] / (len(ib) ** 2)
        Gab = G[np.ix_(ia, ib)] / (len(ia) * len(ib))

        numerator = Ua @ Gab @ Ub.T

        norm_a_sq = np.einsum("ti,ij,tj->t", Ua, Gaa, Ua, optimize=True)
        norm_b_sq = np.einsum("ti,ij,tj->t", Ub, Gbb, Ub, optimize=True)

        norm_a = np.sqrt(np.maximum(norm_a_sq, 0.0))
        norm_b = np.sqrt(np.maximum(norm_b_sq, 0.0))

        denom = norm_a[:, None] * norm_b[None, :]
        corrs = np.divide(
            numerator,
            denom,
            out=np.zeros_like(numerator),
            where=denom > 0,
        )
        corrs = np.nan_to_num(corrs, nan=0.0, posinf=0.0, neginf=0.0)

        m0 = decoder_metrics(corrs)
        m0.update({"rep": rep, "fold": 0})
        rows.append(m0)

        m1 = decoder_metrics(corrs.T)
        m1.update({"rep": rep, "fold": 1})
        rows.append(m1)

    return summarize_metric_rows(rows)



## Fast cumulative top-\(m\) evaluation

Within each held-out evaluation half, archetypes are added in ranked order. The raw
participant reconstructions are accumulated once. At requested \(m\) values, the
current reconstruction is normalized exactly as in the historical decoder and passed
to the batched decoder above.

This avoids recomputing the same matrix products from scratch for every \(m\).


In [ ]:

def temporal_contribution_stack(subjects, archetype):
    """
    Raw rank-1 contribution of one temporal archetype for all subjects.
    Returns N x T x V.
    """
    tvec = np.stack(
        [np.asarray(sub["S"], dtype=float)[archetype, :] for sub in subjects],
        axis=0,
    )  # N x T

    vvec = np.stack(
        [np.asarray(sub["sXC"], dtype=float)[:, archetype] for sub in subjects],
        axis=0,
    )  # N x V

    return tvec[:, :, None] * vvec[:, None, :]


def normalize_stack_over_time(raw_stack):
    mu = raw_stack.mean(axis=1, keepdims=True)
    sd = raw_stack.std(axis=1, keepdims=True)
    return (raw_stack - mu) / (sd + EPS)


def full_temporal_reconstruction_stack(subjects):
    out = []
    for sub in subjects:
        sXC = np.asarray(sub["sXC"], dtype=float)  # V x K
        S = np.asarray(sub["S"], dtype=float)      # K x T
        X = (sXC @ S).T
        X = (X - X.mean(axis=0, keepdims=True)) / (
            X.std(axis=0, keepdims=True) + EPS
        )
        out.append(X)
    return np.stack(out, axis=0)


def evaluate_topm_fast(
    evaluation_subjects,
    ranked_archetypes,
    top_m_values,
    nreps,
    seed,
):
    K = len(ranked_archetypes)
    desired = sorted({int(m) for m in top_m_values if int(m) <= K})
    if not desired:
        return pd.DataFrame()

    # Full model on exactly the same evaluation participants.
    full_stack = full_temporal_reconstruction_stack(evaluation_subjects)
    full_summary = batch_decode_stack(full_stack, nreps=nreps, seed=seed)
    full_mean = full_summary["mean"]

    T = full_stack.shape[1]
    V = full_stack.shape[2]
    chance = 1.0 / T

    raw = np.zeros(
        (len(evaluation_subjects), T, V),
        dtype=float,
    )

    rows = []
    checkpoints = set(desired)
    max_m = max(desired)

    for rank_position, archetype in enumerate(
        ranked_archetypes[:max_m],
        start=1,
    ):
        raw += temporal_contribution_stack(evaluation_subjects, archetype)

        if rank_position not in checkpoints:
            continue

        recon = normalize_stack_over_time(raw)
        summary = batch_decode_stack(
            recon,
            nreps=nreps,
            seed=seed,
        )

        denom = full_mean - chance
        retention = (
            (summary["mean"] - chance) / denom
            if abs(denom) > 1e-12
            else np.nan
        )

        rows.append({
            "top_m": int(rank_position),
            "selected_archetypes": str(
                [int(x) for x in ranked_archetypes[:rank_position]]
            ),
            "mean": summary["mean"],
            "err": summary["err"],
            "std": summary["std"],
            "count": summary["count"],
            "rank_mean": summary["rank_mean"],
            "error_mean": summary["error_mean"],
            "full_mean_same_eval": full_mean,
            "full_err_same_eval": full_summary["err"],
            "retention_above_chance": retention,
            "chance": chance,
        })

    return pd.DataFrame(rows)



## Outer participant splits and resumable storage


In [ ]:

def outer_seed_for(K, condition, outer_rep):
    # Preserve the split definition from the first temporal notebook.
    ss = np.random.SeedSequence([
        int(RNG_SEED),
        1,  # temporal-analysis tag
        int(K),
        int(CONDITION_INDEX[condition]),
        int(outer_rep),
    ])
    return int(ss.generate_state(1, dtype=np.uint32)[0])


def split_outer_subject_indices(n_subjects, outer_seed):
    rng = np.random.default_rng(int(outer_seed))
    idx = np.arange(n_subjects)
    rng.shuffle(idx)
    mid = n_subjects // 2
    return idx[:mid], idx[mid:]


def load_csv(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()


def save_csv(df, path):
    df.to_csv(path, index=False)


def append_dedupe(base, new, keys):
    if new is None or len(new) == 0:
        return base
    out = pd.concat([base, new], ignore_index=True) if len(base) else new.copy()
    return out.drop_duplicates(keys, keep="last").reset_index(drop=True)


seed_rows = []
for K in K_VALUES:
    for condition in CONDITIONS:
        for outer_rep in range(N_OUTER_REPS):
            seed_rows.append({
                "analysis_type": ANALYSIS_TYPE,
                "condition": condition,
                "K": int(K),
                "outer_rep": int(outer_rep),
                "outer_seed": outer_seed_for(K, condition, outer_rep),
            })

pd.DataFrame(seed_rows).to_csv(SEED_CSV, index=False)
print("Saved split table:", SEED_CSV, flush=True)



## Main optimized cross-fit

Progress is printed during both stages:

- `ranking archetype x/K`
- `evaluated outer rep r/20, direction`

Ranking rows are checkpointed every few archetypes. Held-out evaluation rows are
checkpointed after each outer direction.


In [ ]:

run_df = load_csv(RUN_CSV)
rank_df = load_csv(RANK_CSV)

rank_key_cols = [
    "analysis_type", "fit_scope", "condition", "K",
    "outer_rep", "direction", "archetype",
]
run_key_cols = [
    "analysis_type", "fit_scope", "condition", "K",
    "outer_rep", "direction", "top_m",
]

for K in K_VALUES:
    print("\n" + "=" * 90, flush=True)
    print(f"OPTIMIZED TEMPORAL HELD-OUT | K={K}", flush=True)
    print("=" * 90, flush=True)

    for condition in CONDITIONS:
        subjects, source = get_subjects(
            OUTPUT_ROOT,
            ANALYSIS_TYPE,
            FIT_SCOPE,
            int(K),
            condition,
        )

        N = len(subjects)
        if N < 4:
            raise ValueError(f"Need >=4 subjects; got {N}")

        print(
            f"\n{condition} | N={N} | source={source}",
            flush=True,
        )

        # ------------------------------------------------------------
        # Define all outer directions once.
        # ------------------------------------------------------------
        direction_defs = []

        for outer_rep in range(N_OUTER_REPS):
            outer_seed = outer_seed_for(K, condition, outer_rep)
            A_idx, B_idx = split_outer_subject_indices(N, outer_seed)

            direction_defs.extend([
                {
                    "outer_rep": outer_rep,
                    "outer_seed": outer_seed,
                    "direction": "A_to_B",
                    "ranking_idx": A_idx,
                    "evaluation_idx": B_idx,
                    "direction_offset": 0,
                },
                {
                    "outer_rep": outer_rep,
                    "outer_seed": outer_seed,
                    "direction": "B_to_A",
                    "ranking_idx": B_idx,
                    "evaluation_idx": A_idx,
                    "direction_offset": 1,
                },
            ])

        # ------------------------------------------------------------
        # RANKING STAGE
        #
        # Compute each single-archetype factorization once, then score it
        # for every outer ranking half that still needs that archetype.
        # ------------------------------------------------------------
        for archetype in range(int(K)):
            existing = set()

            if len(rank_df):
                cur = rank_df[
                    (rank_df["analysis_type"].astype(str) == ANALYSIS_TYPE)
                    & (rank_df["fit_scope"].astype(str) == FIT_SCOPE)
                    & (rank_df["condition"].astype(str) == condition)
                    & (rank_df["K"].astype(int) == int(K))
                    & (rank_df["archetype"].astype(int) == int(archetype))
                ]

                existing = {
                    (int(r.outer_rep), str(r.direction))
                    for r in cur.itertuples(index=False)
                }

            needed_defs = [
                d for d in direction_defs
                if (int(d["outer_rep"]), str(d["direction"])) not in existing
            ]

            if not needed_defs:
                continue

            U, W, G = temporal_single_factors(subjects, archetype)

            new_rank_rows = []

            for d in needed_defs:
                ranking_seed = (
                    RNG_SEED
                    + 100000 * int(d["outer_rep"])
                    + 1000 * int(K)
                    + int(d["direction_offset"])
                )

                summary = lowrank_single_decode_summary(
                    U,
                    G,
                    d["ranking_idx"],
                    nreps=NREPS_DECODE,
                    seed=ranking_seed,
                )

                new_rank_rows.append({
                    "archetype": int(archetype),
                    "mean_accuracy": summary["mean"],
                    "sem_accuracy": summary["err"],
                    "rank_mean": summary["rank_mean"],
                    "error_mean": summary["error_mean"],
                    "analysis_type": ANALYSIS_TYPE,
                    "fit_scope": FIT_SCOPE,
                    "condition": condition,
                    "K": int(K),
                    "outer_rep": int(d["outer_rep"]),
                    "direction": d["direction"],
                    "outer_seed": int(d["outer_seed"]),
                    "n_ranking_subjects": int(len(d["ranking_idx"])),
                    "fit_source_file": str(source),
                    "fit_source_basename": Path(source).name,
                })

            rank_df = append_dedupe(
                rank_df,
                pd.DataFrame(new_rank_rows),
                rank_key_cols,
            )

            if (
                (archetype + 1) % RANK_PROGRESS_EVERY == 0
                or (archetype + 1) == int(K)
            ):
                save_csv(rank_df, RANK_CSV)
                print(
                    f"  ranking archetype {archetype + 1}/{K}",
                    flush=True,
                )

        save_csv(rank_df, RANK_CSV)

        # ------------------------------------------------------------
        # HELD-OUT EVALUATION STAGE
        # ------------------------------------------------------------
        for d_index, d in enumerate(direction_defs, start=1):
            outer_rep = int(d["outer_rep"])
            direction = str(d["direction"])

            rank_sub = rank_df[
                (rank_df["analysis_type"].astype(str) == ANALYSIS_TYPE)
                & (rank_df["fit_scope"].astype(str) == FIT_SCOPE)
                & (rank_df["condition"].astype(str) == condition)
                & (rank_df["K"].astype(int) == int(K))
                & (rank_df["outer_rep"].astype(int) == outer_rep)
                & (rank_df["direction"].astype(str) == direction)
            ].copy()

            if rank_sub["archetype"].nunique() != int(K):
                raise RuntimeError(
                    f"Incomplete ranking for {condition}/K={K}/"
                    f"rep={outer_rep}/{direction}: "
                    f"{rank_sub['archetype'].nunique()}/{K}"
                )

            ranked = (
                rank_sub
                .sort_values(
                    ["mean_accuracy", "rank_mean", "archetype"],
                    ascending=[False, False, True],
                )["archetype"]
                .astype(int)
                .tolist()
            )

            desired_m = [m for m in TOP_M_VALUES if m <= int(K)]

            existing_m = set()
            if len(run_df):
                cur = run_df[
                    (run_df["analysis_type"].astype(str) == ANALYSIS_TYPE)
                    & (run_df["fit_scope"].astype(str) == FIT_SCOPE)
                    & (run_df["condition"].astype(str) == condition)
                    & (run_df["K"].astype(int) == int(K))
                    & (run_df["outer_rep"].astype(int) == outer_rep)
                    & (run_df["direction"].astype(str) == direction)
                ]
                if len(cur):
                    existing_m = set(cur["top_m"].astype(int))

            missing_m = [m for m in desired_m if m not in existing_m]

            if not missing_m:
                continue

            evaluation_subjects = [
                subjects[int(i)]
                for i in d["evaluation_idx"]
            ]

            evaluation_seed = (
                RNG_SEED
                + 200000 * outer_rep
                + 1000 * int(K)
                + int(d["direction_offset"])
            )

            heldout = evaluate_topm_fast(
                evaluation_subjects,
                ranked,
                top_m_values=missing_m,
                nreps=NREPS_DECODE,
                seed=evaluation_seed,
            )

            heldout["analysis_type"] = ANALYSIS_TYPE
            heldout["fit_scope"] = FIT_SCOPE
            heldout["condition"] = condition
            heldout["K"] = int(K)
            heldout["outer_rep"] = outer_rep
            heldout["direction"] = direction
            heldout["outer_seed"] = int(d["outer_seed"])
            heldout["n_ranking_subjects"] = int(len(d["ranking_idx"]))
            heldout["n_evaluation_subjects"] = int(len(d["evaluation_idx"]))
            heldout["fit_source_file"] = str(source)
            heldout["fit_source_basename"] = Path(source).name

            run_df = append_dedupe(
                run_df,
                heldout,
                run_key_cols,
            )
            save_csv(run_df, RUN_CSV)

            print(
                f"  evaluated {condition} | K={K} | "
                f"outer {outer_rep + 1}/{N_OUTER_REPS} | {direction} | "
                f"m={missing_m}",
                flush=True,
            )

print("\nAll requested temporal configurations complete.", flush=True)
print("Ranking rows:", len(rank_df), flush=True)
print("Held-out rows:", len(run_df), flush=True)



## Outer-split summary

A→B and B→A are two directions of the same outer participant split, so they are
averaged within each outer split before the SEM is computed across the 20 outer splits.


In [ ]:

def sem(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) <= 1 else float(x.std(ddof=1) / np.sqrt(len(x)))

run_df = load_csv(RUN_CSV)

outer = (
    run_df
    .groupby(
        [
            "analysis_type", "fit_scope", "condition",
            "K", "top_m", "outer_rep",
        ],
        as_index=False,
    )
    .agg(
        mean=("mean", "mean"),
        full_mean_same_eval=("full_mean_same_eval", "mean"),
        retention_above_chance=("retention_above_chance", "mean"),
    )
)

summary_df = (
    outer
    .groupby(
        ["analysis_type", "fit_scope", "condition", "K", "top_m"],
        as_index=False,
    )
    .agg(
        mean=("mean", "mean"),
        err=("mean", sem),
        std=("mean", "std"),
        n_outer=("mean", "size"),
        mean_full_same_eval=("full_mean_same_eval", "mean"),
        mean_retention=("retention_above_chance", "mean"),
        err_retention=("retention_above_chance", sem),
    )
)

summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df.head(30))

coverage = (
    summary_df
    .groupby(["condition", "K"])["top_m"]
    .agg(lambda x: sorted(set(map(int, x))))
    .reset_index(name="top_m_available")
)

display(coverage)
print("Saved:", SUMMARY_CSV, flush=True)



## Optional equivalence spot-check

The optimized decoder was developed by testing it against the original
`decode_subjects` implementation on synthetic data for both spatial and temporal
orientations, with exact agreement in accuracy, rank, and error.

If desired, the cell below can be adapted to compare one real-data configuration
against the generic decoder. It is left commented because the purpose of this notebook
is to avoid the generic repeated reconstruction bottleneck.
